In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from astropy.io import fits

In [9]:
import numpy as np
import csv
from astropy.io import fits

# -----------------------------
# File paths
# -----------------------------
fits_file = r"C:/Users/jshld/Downloads/LOPS2PICtarget2.1.0.1-t-fg-c-scv.fits"
csv_file  = r"C:/Users/jshld/Downloads/Adjusted final list.csv"

# -----------------------------
# Helper: read CSV into list
# -----------------------------
def list_import(path):
    with open(path, "r", newline="", encoding="utf-8-sig") as f:
        return list(csv.reader(f))

# -----------------------------
# Helper: convert FITS strings to normal Python strings
# -----------------------------
def fits_col_to_str_array(col):
    out = []
    for x in col:
        if isinstance(x, (bytes, bytearray)):
            out.append(x.decode("utf-8", errors="ignore").strip())
        else:
            out.append(str(x).strip())
    return np.array(out, dtype=object)

# -----------------------------
# Load CSV
# Assumes first row may be a header; skips it if needed
# -----------------------------
star_list = list_import(csv_file)

# Optional header detection (simple heuristic)
start_idx = 1 if len(star_list) > 0 and not star_list[0][0].startswith("Gaia") else 0

# Build list of requested PIC names from CSV first column
csv_picnames = []
star_labels = []   # stores the f"{mystar[0]}: M = {mystar[1]}, R = {mystar[2]}" strings

for i in range(start_idx, len(star_list)):
    mystar = star_list[i]
    if len(mystar) < 3:
        continue  # skip malformed rows

    picname = str(mystar[0]).strip()
    csv_picnames.append(picname)

    # Build the label you mentioned (if columns 1 and 2 exist)
    try:
        star_name = f"{mystar[0]}: M = {mystar[1]}, R = {mystar[2]} Gmag = {mystar[4]}"
    except IndexError:
        star_name = str(mystar[0])
    star_labels.append(star_name)

# -----------------------------
# Open FITS and extract columns
# -----------------------------
with fits.open(fits_file) as hdul:
    data = hdul[1].data  # assumes table is in extension 1

    # Check required columns exist
    required = ["PICname", "Gmag", "BOLrandomSysNSRNCAM_T"]
    missing = [c for c in required if c not in data.columns.names]
    if missing:
        raise KeyError(f"Missing required FITS columns: {missing}")

    # Convert PICname column to clean strings for matching
    fits_picnames = fits_col_to_str_array(data["PICname"])

    # Build a lookup: PICname -> row index
    # (if duplicates exist, this keeps the first occurrence)
    pic_to_index = {}
    for idx, name in enumerate(fits_picnames):
        if name not in pic_to_index:
            pic_to_index[name] = idx

    # Output arrays
    Gmag_array = []
    BOLrandomSysNSRNCAM_T_array = []

    # Optional bookkeeping
    matched_picnames = []
    matched_star_labels = []
    not_found = []

    # Match CSV entries to FITS rows and extract values
    for picname, star_label in zip(csv_picnames, star_labels):
        idx = pic_to_index.get(picname)

        if idx is None:
            not_found.append(picname)
            continue

        gmag_val = data["Gmag"][idx]
        bol_val  = data["BOLrandomSysNSRNCAM_T"][idx]

        # Convert numpy scalars -> Python scalars if needed
        if isinstance(gmag_val, np.generic):
            gmag_val = gmag_val.item()
        if isinstance(bol_val, np.generic):
            bol_val = bol_val.item()

        Gmag_array.append(gmag_val)
        BOLrandomSysNSRNCAM_T_array.append(bol_val)
        matched_picnames.append(picname)
        matched_star_labels.append(star_label)

# Convert to numpy arrays
Gmag_array = np.array(Gmag_array, dtype=float)
BOLrandomSysNSRNCAM_T_array = np.array(BOLrandomSysNSRNCAM_T_array, dtype=float)

# -----------------------------
# Results
# -----------------------------
print(f"Matched {len(Gmag_array)} stars")
print(f"Not found in FITS: {len(not_found)}")

if not_found:
    print("\nFirst few not found:")
    for x in not_found[:10]:
        print("  ", x)

# These are your two separate arrays:
print("\nPIC_IDs:")
print(matched_picnames)

print("\nBOLrandomSysNSRNCAM_T_array:")
print(BOLrandomSysNSRNCAM_T_array)

DF1 = pd.DataFrame({"PIC_ID": matched_picnames,
                    "sig1hr": BOLrandomSysNSRNCAM_T_array,
                    "Gmag": Gmag_array
                   })
print(DF1)

Matched 33 stars
Not found in FITS: 0

PIC_IDs:
['PIC 2915577000081', 'PIC 2915571000104', 'PIC 2912861000050', 'PIC 2908735000056', 'PIC 2908735000057', 'PIC 2901793000033', 'PIC 2897612000102', 'PIC 2893330000017', 'PIC 2890529000062', 'PIC 2889072000139', 'PIC 2886168000063', 'PIC 2886166000078', 'PIC 2886218000135', 'PIC 2886191000129', 'PIC 2883305000009', 'PIC 2880424000123', 'PIC 2880424000128', 'PIC 2878914000067', 'PIC 2878966000167', 'PIC 2877464000062', 'PIC 2874507000079', 'PIC 2870094000052', 'PIC 2868606000109', 'PIC 2868638000148', 'PIC 2864154000102', 'PIC 2862637000040', 'PIC 2861112000105', 'PIC 2856556000125', 'PIC 2847353000081', 'PIC 2844331000233', 'PIC 2841143000066', 'PIC 2833293000069', 'PIC 2705309000027']

BOLrandomSysNSRNCAM_T_array:
[19.51000023 18.09000015 25.93000031 20.35000038 24.15999985 16.28000069
 19.23999977 26.70999908 11.36999989 21.88999939 21.13999939 19.46999931
 23.17000008 25.35000038 25.18000031 17.17000008 19.87000084 15.85000038
 23.34000

In [12]:
def shot_noise(star_name, power):
    power = np.asarray(power).squeeze()  # make sure it's 1D

    mask = (DF1["PIC_ID"].astype(str) == str(star_name))
    sig_series = DF1.loc[mask, "sig1hr"]

    if sig_series.empty:
        raise KeyError(f"No match for PIC_ID={star_name}")
    if len(sig_series) > 1:
        raise ValueError(f"Multiple matches for PIC_ID={star_name}")

    sig1hr = float(sig_series.iloc[0])  # scalar

    bg_noise = 2 * (sig1hr**2) * 3600 * 1e-6
    spectrum = power + bg_noise

    noisy_spec = []
    for x in spectrum:
        a = np.random.uniform(0, 1)
        noisy_spec.append(x * (-np.log(a)))

    return np.array(noisy_spec)
